# Redundant Temperature Sensor Analysis

Fuse three cheap temperature sensors (NTC thermistor, TMP36, MCP9700A) and compare the result against a TMP117 reference.

1. Look at the raw readings and their spread vs. the TMP117
2. Naive fusion: a plain average of the raw readings
3. Calibrate: measure each sensor's bias and noise
4. Inverse-variance weighting
5. Test both fusion methods on a fresh recording

The TMP117 is factory calibrated (±0.1 °C from −20 to 50 °C) and its noise is tiny compared to the other sensors, so we treat it as ground truth.

**Two recordings:** offsets and noise levels are measured on a *calibration* log, and the fusion is tested on a separate *evaluation* log. If you calibrated and tested on the same data, the bias would come out as zero by construction.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

## Settings

In [ ]:
# Recordings (folders containing temp.csv and meta.json). A name without the
# timestamp suffix picks the newest matching folder, so "recordings/static_cal"
# matches "recordings/static_cal-20261020-143000".
CAL_RECORDING = "recordings/static_cal"    # Used to measure offsets and noise
EVAL_RECORDING = "recordings/static_eval"  # Fresh log to test the fusion on

# Cheap sensors (CSV column -> label) and the reference column
SENSORS = {
    "ntc_c": "NTC",
    "tmp36_c": "TMP36",
    "mcp9700_c": "MCP9700A",
}
REF = "tmp117_c"

# Plot colors
COLORS = {
    "ntc_c": "tab:orange",
    "tmp36_c": "tab:blue",
    "mcp9700_c": "tab:purple",
    "avg": "tab:gray",
    "ivw": "tab:green",
    REF: "k",
}

# Histogram bin widths (deg C). The TMP36 and MCP9700A read in steps of one
# ADC count (3.3 V / 4095 * 100 = 0.08 C), so error bins are half a step wide
# to avoid comb-like artifacts.
BIN_C = 0.02
ERR_BIN_C = 3.3 / 4095 * 100 / 2

## Helper Functions

In [ ]:
def find_recording(path):
    """Return the folder, or the newest '<path>-<timestamp>' folder."""
    p = Path(path)
    if p.is_dir():
        return p
    matches = sorted(m for m in p.parent.glob(f"{p.name}-*") if m.is_dir())
    if not matches:
        raise FileNotFoundError(f"No recording found for {path}")
    return matches[-1]


def load_recording(path):
    """
    Load temp.csv and meta.json from a temp_logger recording. Returns
    (rec, meta), where rec is a dict of float arrays keyed by column name,
    plus 't' in seconds. Rows without a TMP117 reading are dropped.
    """
    folder = find_recording(path)
    with open(folder / "meta.json", encoding="utf-8") as f:
        meta = json.load(f)

    # Read the numeric columns ("nan" parses as NaN)
    with open(folder / "temp.csv", encoding="utf-8") as f:
        header = f.readline().strip().split(",")
    cols = [c for c in header if c != "tag"]
    data = np.genfromtxt(folder / "temp.csv", delimiter=",", skip_header=1,
                         usecols=[header.index(c) for c in cols], ndmin=2)
    rec = {c: data[:, i] for i, c in enumerate(cols)}
    rec["t"] = rec["t_us"] * 1e-6

    # Drop rows with no reference reading
    keep = ~np.isnan(rec[REF])
    if (~keep).any():
        print(f"Dropped {(~keep).sum()} row(s) with no TMP117 reading")
    rec = {k: v[keep] for k, v in rec.items()}

    print(f"Loaded {folder}: {len(rec['t'])} samples over "
          f"{rec['t'][-1] - rec['t'][0]:.0f} s")
    return rec, meta


def error_stats(x, ref):
    """Bias (mean error) and noise (std of error) vs. the reference."""
    e = x - ref
    return np.nanmean(e), np.nanstd(e, ddof=1)


def noise_from_diffs(x):
    """
    Noise sigma from sample-to-sample differences. Slow drift cancels out,
    and white noise with sigma s gives differences with sigma s * sqrt(2).
    Used for the TMP117, which has no better reference to compare against.
    """
    return np.nanstd(np.diff(x), ddof=1) / np.sqrt(2)


def print_table(rows, title):
    """rows = [(label, bias, sigma, predicted_sigma)]; None prints a dash."""
    fmt = lambda v, spec: "—" if v is None else format(v, spec)
    print(title)
    print(f"  {'':<14}{'bias (°C)':>11}{'σ (°C)':>10}{'predicted σ':>14}")
    for label, b, s, p in rows:
        print(f"  {label:<14}{fmt(b, '+.3f'):>11}{fmt(s, '.3f'):>10}"
              f"{fmt(p, '.3f'):>14}")
    print()


def plot_series(t, series, title):
    """Plot temperature traces. series = [(label, values, color), ...]"""
    fig, ax = plt.subplots(figsize=(10, 4))
    for label, y, color in series:
        ax.plot(t, y, label=label, color=color, lw=2.0 if color == "k" else 1.0)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Temperature (°C)")
    ax.set_title(title)
    ax.legend(loc="upper right", ncol=len(series))
    ax.grid(alpha=0.3)
    plt.show()


def plot_histograms(rows, ref_value, title, xlabel, bin_c=BIN_C):
    """
    Stacked histograms on a shared x axis, with a dashed reference line.
    rows = [(label, values, color), ...]
    """
    values = [v[~np.isnan(v)] for _, v, _ in rows]
    lo = min(min(v.min() for v in values), ref_value)
    hi = max(max(v.max() for v in values), ref_value)
    bins = np.arange(np.floor(lo / bin_c) * bin_c, hi + 2 * bin_c, bin_c)

    fig, axes = plt.subplots(len(rows), 1, figsize=(10, 1.6 * len(rows)),
                             sharex=True)
    for ax, (label, _, color), v in zip(np.atleast_1d(axes), rows, values):
        ax.hist(v, bins=bins, color=color, alpha=0.85)
        ax.axvline(ref_value, color="k", ls="--", lw=1)
        ax.set_ylabel(label, rotation=0, ha="right", va="center")
        ax.set_yticks([])
        ax.text(0.99, 0.9, f"mean = {v.mean():.3f}   σ = {v.std(ddof=1):.3f}",
                transform=ax.transAxes, ha="right", va="top", fontsize=9)
    np.atleast_1d(axes)[-1].set_xlabel(xlabel)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

## Look at the Raw Readings

Plot all four sensors on the calibration log. Each cheap sensor sits at a different offset from the TMP117, and some are visibly noisier than others.

In [ ]:
# Load both recordings
cal, cal_meta = load_recording(CAL_RECORDING)
ev, ev_meta = load_recording(EVAL_RECORDING)
if find_recording(CAL_RECORDING) == find_recording(EVAL_RECORDING):
    print("WARNING: calibration and evaluation use the same log, so the "
          "fused bias will be zero by construction")

In [ ]:
# Time series of the calibration log
plot_series(
    cal["t"],
    [(SENSORS[k], cal[k], COLORS[k]) for k in SENSORS] + [("TMP117", cal[REF], "k")],
    "Calibration log: raw readings",
)

### Histograms

The dashed line is the TMP117 mean. The distance from the line to a histogram's center is that sensor's **bias** (systematic error), and the histogram's width is its **noise** (random error).

The σ shown here is the spread of the raw readings, so it also includes any slow room drift during the log. The calibration step below measures noise against the TMP117 instead, which cancels drift.

In [ ]:
plot_histograms(
    [(SENSORS[k], cal[k], COLORS[k]) for k in SENSORS] + [("TMP117", cal[REF], "k")],
    ref_value=np.mean(cal[REF]),
    title="Calibration log: raw readings vs. TMP117",
    xlabel="Temperature (°C)",
)

## Naive Fusion: Plain Average

Average the three raw readings and compare against the TMP117. Averaging smooths out random noise, but each sensor's bias goes straight through: the average's bias is just the mean of the individual biases.

In [ ]:
# Plain average of the raw readings
avg_raw = np.mean([cal[k] for k in SENSORS], axis=0)

plot_histograms(
    [("Average (raw)", avg_raw, COLORS["avg"]), ("TMP117", cal[REF], "k")],
    ref_value=np.mean(cal[REF]),
    title="Calibration log: plain average of raw readings vs. TMP117",
    xlabel="Temperature (°C)",
)

rows = [(SENSORS[k], *error_stats(cal[k], cal[REF]), None) for k in SENSORS]
rows.append(("Average (raw)", *error_stats(avg_raw, cal[REF]), None))
print_table(rows, "Calibration log, error vs. TMP117")

## Calibrate: Remove Each Sensor's Bias

No amount of averaging removes a constant offset, so we measure each sensor's offset against the TMP117 and subtract it. This is a one-point calibration: it fixes the offset at this temperature, but not a slope (gain) error.

We work with **errors** (sensor − TMP117) rather than raw readings. Slow room drift moves every sensor together, so it cancels in the error:

- Mean of the error = offset to subtract
- Standard deviation of the error = noise σ, used for the weights in the next step

In [ ]:
# Offsets and noise from the calibration log
offset, sigma = {}, {}
for k in SENSORS:
    offset[k], sigma[k] = error_stats(cal[k], cal[REF])

rows = [(SENSORS[k], offset[k], sigma[k], None) for k in SENSORS]
rows.append(("TMP117", None, noise_from_diffs(cal[REF]), None))
print_table(rows, "Calibration log, error vs. TMP117 (bias = offset to subtract)")


def calibrate(rec):
    """Subtract each sensor's calibration offset."""
    return {k: rec[k] - offset[k] for k in SENSORS}

## Inverse-Variance Weighting

A plain average gives every sensor the same vote, so the noisiest sensor drags the result around. For $N$ sensors with independent noise $\sigma_i$, the plain average has

$$\sigma_{avg} = \frac{\sqrt{\sigma_1^2 + \sigma_2^2 + \dots + \sigma_N^2}}{N}$$

Inverse-variance weighting gives each sensor a weight proportional to $1/\sigma_i^2$:

$$w_i = \frac{1/\sigma_i^2}{\sum_j 1/\sigma_j^2} \qquad \hat{T} = \sum_i w_i T_i$$

The fused estimate has

$$\sigma_{fused} = \frac{1}{\sqrt{\sum_j 1/\sigma_j^2}}$$

which is always lower than the best single sensor. Adding a worse sensor still helps, as long as it's weighted correctly.

This assumes each sensor has **zero bias** (we just calibrated) and **independent noise** (checked at the end).

In [ ]:
# Weights from the calibration-log noise
inv_var = {k: 1.0 / sigma[k] ** 2 for k in SENSORS}
total = sum(inv_var.values())
weight = {k: inv_var[k] / total for k in SENSORS}

# Predicted noise of each method
n = len(SENSORS)
sigma_avg_pred = np.sqrt(sum(sigma[k] ** 2 for k in SENSORS)) / n
sigma_ivw_pred = 1.0 / np.sqrt(total)
best = min(SENSORS, key=lambda k: sigma[k])

print("Weights")
for k in SENSORS:
    print(f"  {SENSORS[k]:<10} σ = {sigma[k]:.3f} °C   weight = {weight[k]:.1%}")
print()
print(f"Predicted σ, plain average:        {sigma_avg_pred:.3f} °C")
print(f"Predicted σ, inverse-variance:     {sigma_ivw_pred:.3f} °C")
print(f"Best single sensor ({SENSORS[best]}):" .ljust(35) + f"{sigma[best]:.3f} °C")

## Test on a Fresh Recording

Apply the offsets and weights from the calibration log to the evaluation log, and compare:

- Each calibrated sensor on its own
- Plain average of the calibrated sensors
- Inverse-variance weighted average

In [ ]:
# Calibrate the evaluation log and fuse it
ev_cal = calibrate(ev)
avg_cal = np.mean([ev_cal[k] for k in SENSORS], axis=0)
ivw = sum(weight[k] * ev_cal[k] for k in SENSORS)

rows = [(SENSORS[k], *error_stats(ev_cal[k], ev[REF]), sigma[k]) for k in SENSORS]
rows.append(("Average", *error_stats(avg_cal, ev[REF]), sigma_avg_pred))
rows.append(("Inv-var", *error_stats(ivw, ev[REF]), sigma_ivw_pred))
rows.append(("TMP117", None, noise_from_diffs(ev[REF]), None))
print_table(rows, "Evaluation log, error vs. TMP117 (after calibration)")

In [ ]:
# Error histograms: centered on zero after calibration, narrower is better
plot_histograms(
    [(SENSORS[k], ev_cal[k] - ev[REF], COLORS[k]) for k in SENSORS]
    + [("Average", avg_cal - ev[REF], COLORS["avg"]),
       ("Inv-var", ivw - ev[REF], COLORS["ivw"])],
    ref_value=0.0,
    title="Evaluation log: error vs. TMP117 (after calibration)",
    xlabel="Error (°C)",
    bin_c=ERR_BIN_C,
)

In [ ]:
# Summary: measured noise of each sensor and fusion method
labels = [SENSORS[k] for k in SENSORS] + ["Average", "Inv-var", "TMP117"]
sigmas = [error_stats(ev_cal[k], ev[REF])[1] for k in SENSORS]
sigmas += [error_stats(avg_cal, ev[REF])[1], error_stats(ivw, ev[REF])[1],
           noise_from_diffs(ev[REF])]
colors = [COLORS[k] for k in SENSORS] + [COLORS["avg"], COLORS["ivw"], "k"]

fig, ax = plt.subplots(figsize=(8, 3.5))
bars = ax.barh(labels[::-1], sigmas[::-1], color=colors[::-1])
ax.bar_label(bars, fmt="%.3f", padding=3)
ax.set_xlabel("Noise σ vs. TMP117 (°C)")
ax.set_title("Evaluation log: lower is better")
ax.set_xlim(0, max(sigmas) * 1.2)
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Time series: best single sensor vs. fused vs. TMP117
plot_series(
    ev["t"],
    [(SENSORS[best], ev_cal[best], COLORS[best]),
     ("Average", avg_cal, COLORS["avg"]),
     ("Inv-var", ivw, COLORS["ivw"]),
     ("TMP117", ev[REF], "k")],
    "Evaluation log: calibrated and fused readings",
)

## Check the Independence Assumption

Inverse-variance weighting assumes each sensor's noise is independent. The TMP36 and MCP9700A both read through the same ADC and depend on the same 3.3 V rail, so rail noise shows up in both at once. (The NTC divider is ratiometric, so rail noise cancels for it.)

A correlation near 0 means the noise is independent. If two sensors are clearly correlated (above ~0.2), their errors don't average out, and the measured fused σ comes out higher than predicted.

In [ ]:
# Correlation between sensor errors on the calibration log
errs = np.array([cal[k] - cal[REF] for k in SENSORS])
corr = np.corrcoef(errs)

names = [SENSORS[k] for k in SENSORS]
print("Error correlation (calibration log)")
print("  " + " " * 10 + "".join(f"{nm:>10}" for nm in names))
for nm, row in zip(names, corr):
    print(f"  {nm:<10}" + "".join(f"{c:>10.2f}" for c in row))